# 06 Multivariable Analysis — Reference Solutions

Use the Songbai Nursing Home Legionnaires' disease line list to practice Modified Poisson regression (adjusted RR)
and logistic regression (adjusted OR), and compare the two.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (avoid Chinese labels rendering as boxes) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- Load the data ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## Question 1: Predicting death — Crude RR vs Crude OR

1. Build the `dead` column
2. Compute the case fatality rate
3. Compute the crude RR (Modified Poisson) and crude OR (logistic) at the same time
4. Organize into a comparison table

In [ ]:
# --- Build the outcome variable ---
df["dead"] = (df["outcome"] == "dead").astype(int)

# Convert severity to numbers (non-infected set to 0)
sev_map = {"not_ill": 0, "asymptomatic": 0, "mild": 1, "moderate": 2, "severe": 3}
df["severity_score"] = df["clinical_severity"].map(sev_map)

# Use only the infected for death prediction (non-infected won't die of this disease)
cases = df[df["infected"] == 1].copy()
cfr = cases["dead"].mean()
print(f"Infected: {len(cases)} people, deaths: {cases['dead'].sum()} people")
print(f"Case fatality rate (CFR): {cfr:.1%}")
print(f"→ CFR = {cfr:.1%}, much lower than the 43% attack rate")
print(f"→ We expect the gap between OR and RR to be smaller than when predicting infection\n")

# --- Compute crude RR and crude OR at the same time ---
factors_death = ["age", "comorbidity_chf", "comorbidity_copd",
                 "immunosuppressed", "severity_score"]

crude_rows = []
for var in factors_death:
    # Modified Poisson → crude RR
    poisson = smf.glm(
        f"dead ~ {var}", data=cases,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0", disp=0)
    rr = np.exp(poisson.params[var])
    rr_ci = np.exp(poisson.conf_int().loc[var])

    # Logistic → crude OR
    logit = smf.logit(f"dead ~ {var}", data=cases).fit(disp=0)
    or_val = np.exp(logit.params[var])
    or_ci = np.exp(logit.conf_int().loc[var])

    crude_rows.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}–{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}–{or_ci[1]:.3f}",
    })

crude_df = pd.DataFrame(crude_rows)
print("=== Death prediction: Crude RR vs Crude OR ===")
print(crude_df.to_string(index=False))
print("\n→ With a lower fatality rate (~16%), the gap between OR and RR is much smaller than for infection prediction (43%)")

## Question 2: Multivariable Adjusted RR + Adjusted OR

Build a multivariable model predicting death, using both Modified Poisson and Logistic Regression,
and compare the adjusted RR and adjusted OR side by side.

In [ ]:
# --- Shared formula ---
formula_death = (
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score"
)

# --- Modified Poisson → Adjusted RR ---
poisson_multi = smf.glm(
    formula_death, data=cases,
    family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Logistic → Adjusted OR ---
logit_multi = smf.logit(formula_death, data=cases).fit(disp=0, method="lbfgs")

# --- Side-by-side comparison table ---
compare_rows = []
for var in poisson_multi.params.index:
    if var == "Intercept":
        continue
    # Adjusted RR
    rr = np.exp(poisson_multi.params[var])
    rr_ci = np.exp(poisson_multi.conf_int().loc[var])
    # Adjusted OR
    or_val = np.exp(logit_multi.params[var])
    or_ci = np.exp(logit_multi.conf_int().loc[var])
    # How much the OR overestimates relative to the RR
    pct_diff = (or_val - rr) / rr * 100

    compare_rows.append({
        "variable": var,
        "adj_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}–{rr_ci[1]:.3f}",
        "adj_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}–{or_ci[1]:.3f}",
        "OR overest.%": f"{pct_diff:+.1f}%",
    })

compare_df = pd.DataFrame(compare_rows)
print("=== Death prediction: Adjusted RR vs Adjusted OR ===")
print(compare_df.to_string(index=False))

# --- Crude vs Adjusted comparison ---
print("\n=== Crude → Adjusted change (using RR) ===")
for var in factors_death:
    c_row = crude_df[crude_df["variable"] == var].iloc[0]
    a_row = compare_df[compare_df["variable"] == var]
    if len(a_row) == 0:
        continue
    a_row = a_row.iloc[0]
    change = (a_row["adj_RR"] - c_row["crude_RR"]) / c_row["crude_RR"] * 100
    print(f"  {var:25s}  crude_RR={c_row['crude_RR']:.3f}  "
          f"adj_RR={a_row['adj_RR']:.3f}  ({change:+.1f}%)")
print("\n→ The variable with the largest change = the factor most confounded by the others")

## Question 3 (challenge): Model comparison + Forest Plot

1. Build two Modified Poisson models (reduced vs full)
2. Compare the AIC
3. Draw an Adjusted RR forest plot using the better model

In [ ]:
# --- Model A (reduced): 3 predictors ---
model_a = smf.glm(
    "dead ~ age + immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Model B (full): 5 predictors ---
model_b = smf.glm(
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- AIC comparison ---
print("=== Model comparison (Modified Poisson) ===")
print(f"  Model A (3 variables) AIC = {model_a.aic:.1f}")
print(f"  Model B (5 variables) AIC = {model_b.aic:.1f}")

best = model_a if model_a.aic < model_b.aic else model_b
best_name = "A" if model_a.aic < model_b.aic else "B"
print(f"  → Model {best_name} is better (smaller AIC = the best balance of explanatory power and parsimony)")

In [ ]:
# --- Forest Plot: Adjusted RR (using the better model) ---
forest_data = []
for var in best.params.index:
    if var == "Intercept":
        continue
    rr = np.exp(best.params[var])
    ci = np.exp(best.conf_int().loc[var])
    forest_data.append({
        "variable": var,
        "RR": rr,
        "ci_lo": ci[0],
        "ci_hi": ci[1],
    })

fdf = pd.DataFrame(forest_data)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(fdf))

# Point estimate + confidence interval
ax.errorbar(
    fdf["RR"], y_pos,
    xerr=[fdf["RR"] - fdf["ci_lo"], fdf["ci_hi"] - fdf["RR"]],
    fmt="o", color="#D97757", capsize=4, markersize=8,
    ecolor="#6A9BCC", elinewidth=2,
)

# RR = 1 reference line (no effect)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5, label="RR = 1")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(fdf["variable"])
ax.set_xlabel("Adjusted Risk Ratio (RR)")
ax.set_title(f"Death prediction model {best_name} — Adjusted RR forest plot")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- Interpretation ---
print("\n=== Independent predictors (RR > 1 and CI does not include 1) ===")
for _, row in fdf.iterrows():
    sig = "✓ significant" if row["ci_lo"] > 1 else "  not significant"
    print(f"  {row['variable']:25s}  RR={row['RR']:.3f}  "
          f"({row['ci_lo']:.3f}–{row['ci_hi']:.3f})  {sig}")

### Interpretation

- **severity_score**: clinical severity is the strongest predictor of death (largest RR), which matches intuition
- **immunosuppressed**: after controlling for severity, immunosuppression may still be an independent risk factor
- **age**: the RR per one-year increase in age looks close to 1, but the cumulative effect is large (e.g., 80 vs 70 is a 10-year difference)
- **RR vs OR**: with a ~16% fatality rate, the gap between OR and RR is smaller than for infection prediction (43% attack rate), confirming the principle "the lower the prevalence, the closer the OR is to the RR"
- **Model selection**: the model with the smaller AIC won't necessarily have every variable significant, but it has a better overall balance
- **Limitation**: there are only ~19 deaths, so the model has limited degrees of freedom and shouldn't include too many variables